# Finetune Ψ₀ + SONIC 1.1 trên Colab: task can_pnp (ngồi xổm lấy lon trên bục → đi tới bàn → đặt vào khay)

Notebook này dựng môi trường, nạp dữ liệu tự sinh và checkpoint warm-start **`postpre.sonic1.1.unifolm.2609181726.40k`**, rồi finetune Ψ₀
(VLM đóng băng, chỉ train action header; SONIC 1.1; không RTC lúc train). Dữ liệu và checkpoint nằm trên **Google Drive của chính tài khoản đang chạy Colab**.

**Khác với notebook `box_pnp` (SONIC 1.0):**

| | `box_pnp/colab_finetune_psi0.ipynb` | Notebook này |
|---|---|---|
| Token | SONIC 1.0 | **SONIC 1.1** (dữ liệu sinh bằng `sonic_v1_1`) |
| Warm-start | `sonic-wbcbox` (sim, 6 khối) | `postpre.sonic1.1.unifolm` (post-train UniFoLM G1, 12 khối) |
| Action / state | 78 / 43 | 78 / 43 trong dữ liệu, **tự đệm 0 lên 80 / 45** lúc nạp (2 chiều đệm không tính vào loss) |
| Model | cũ | `combined-temb` + câu lệnh qua CLIP, `qk-norm`, state làm token, token null khi bỏ state |
| Code Ψ₀ | `4f3720d` | `2830f93` (bản phát hành checkpoint 1.1) |

Cách đệm 80/45 giống hệt lúc checkpoint được post-train trên UniFoLM (`scripts/train/psi0/posttrain-psix-unifolm-g1-sonic1.1.sh`):
UniFoLM cũng là G1 không cổ, action `token 64 + tay 14`, state 43 theo thứ tự khớp 0810, ảnh 320×240.

**Chuẩn bị (một lần):**

1. Export dữ liệu trên máy local:
   ```
   .venv-datagen/bin/python scripts/datagen/can_pnp/export_psi0.py .data/can_ped_144 --out .data/psi0 --zip
   ```
2. Đăng nhập Colab bằng tài khoản Google sẽ dùng, rồi upload `.data/psi0/can_ped_144_psi0.zip` vào **My Drive/psi0/** của tài khoản đó
   (đường dẫn đặt ở `DATA_ZIP`, cell 1).
3. Drive cần **trống ít nhất 40 GB**: header 12 khối làm checkpoint lớn hơn bản `box_pnp` (ước tính khoảng 15–17 GB mỗi checkpoint,
   smoke test ở cell 8 in ra con số thật), và lúc tải checkpoint mới lên thì checkpoint cũ vẫn còn trên Drive.

**Runtime:** `Runtime → Change runtime type` → GPU **G4 (RTX PRO 6000, 96 GB), H100 hoặc A100**, bật **High-RAM**. L4 (24 GB) có thể
chạy với batch rất nhỏ nhưng chậm; T4 không dùng được (không có bf16). Cell 3 tự chọn bộ cài theo GPU. **Lần đầu chạy, hãy chạy smoke test (cell 8).**

**Colab Secrets** (biểu tượng chìa khoá ở thanh bên trái, bật "Notebook access"): `WANDB_API_KEY` (tuỳ chọn; không có thì W&B chạy offline).

| Cell | Việc | Lần đầu | Sau khi máy Colab bị thu hồi |
|---|---|---|---|
| 1 | Cấu hình | ✅ | ✅ |
| 2 | GPU, kết nối Drive (hiện hộp thoại cấp quyền), secrets, mã run | ✅ | ✅ |
| 3 | Cài môi trường (`.venv-psi`: Python 3.11, torch 2.7, flash-attn), 10–15 phút | ✅ | ✅ |
| 4 | Ghi `.env` | ✅ | ✅ |
| 5 | Tải và kiểm tra dữ liệu | ✅ | ✅ |
| 6 | Tải checkpoint warm-start (11,2 GB) và kiểm tra nó khớp với lệnh train, 5–10 phút | ✅ | ✅ |
| 7 | Ghi script train | ✅ | ✅ |
| 8 | Smoke test 40 bước: đo VRAM, tốc độ và ước tính thời gian | ✅ | bỏ qua |
| 9 | **Train, để cell chạy suốt** | ✅ | ✅ (tự resume) |
| 10 | Dừng train | chỉ khi muốn dừng | như cột trước |

- Cell 1 chỉ khai báo cấu hình; cell 2 mới đưa nó vào môi trường cho các cell sau. **Sau khi sửa cell 1, hoặc sau khi runtime
  khởi động lại, luôn chạy lại cell 1 rồi cell 2.**
- **Cell 9 phải luôn ở trạng thái đang chạy.** Train chạy ở một tiến trình nền, còn cell 9 hiển thị tiến độ mỗi phút và tải từng checkpoint
  lên Drive. Colab thu hồi máy của notebook không có cell nào đang chạy sau khoảng 90 phút, kể cả khi train vẫn chạy nền.
  Lỡ dừng cell 9 thì train vẫn tiếp tục; chạy lại cell 9 để theo dõi và tải checkpoint tiếp.
- Giữ tab Colab mở và máy tính không ngủ. Chỉ Colab Pro+ mới chạy tiếp khi đóng trình duyệt.
- **Khi máy Colab bị thu hồi:** mở lại notebook, chạy cell 1 → 7 rồi cell 9. Cell 9 tải checkpoint mới nhất từ Drive về và train tiếp
  từ đó. Muốn bắt đầu một run mới thì đổi `EXP`.

In [ ]:
# @title 1. Cấu hình
# Only declares the settings: cell 2 exports them (plus the values it derives) for the other cells,
# so after editing this cell run cell 2 as well.
CFG = {
    # code: the Psi0 release that ships the SONIC 1.1 checkpoint (its train flags and the 80/45 padding are taken from it)
    "PSI0_REPO": "https://github.com/physical-superintelligence-lab/Psi0.git",
    "PSI0_COMMIT": "2830f9367fc12ca49d9b4df4348726daae9db15d",
    # on the Google Drive of the account running this notebook, relative to My Drive
    "DATA_ZIP": "psi0/can_ped_144_psi0.zip",  # the zip written by export_psi0.py --zip
    "DRIVE_RUNS": "psi0/runs",                      # checkpoints, logs and the run id
    "PACK": "can_ped_144",
    # run
    "EXP": "canped-s11",
    "RUN_TS": "",           # empty: reuse <DRIVE_RUNS>/<EXP>.run_ts (auto-resume), else start one
    "GLOBAL_BATCH": "128",  # BS x ACC, kept constant across GPUs
    "BS": "",               # per-step batch; empty = pick from VRAM in cell 2
    "MAX_STEPS": "15000",   # optimizer steps; cell 8 prints the time per step and the estimate
    "WARMUP": "500",
    "CKPT_EVERY": "1000",   # at most this much training is lost if the VM goes away
    "VAL_EVERY": "1000",
    "VAL_BATCHES": "5",
    "KEEP_CKPTS": "2",      # on the Colab disk; Drive keeps only the newest complete checkpoint
    "STATE_DROP_PROB": "0.1",  # per-sample state drop -> the checkpoint's learned null token (the 1.1 finetune recipe)
    "EXTRA_ARGS": "",       # appended to the train command, e.g. ROBUST_ARGS below
}

# robustness knobs of the Psi0-SONIC release (examples/psi0_for_sonic.md, items 8-9)
ROBUST_ARGS = ("--data.transform.repack.state-temporal-jitter=10 "
               "--data.transform.repack.state-temporal-jitter-prob=0.5 "
               "--data.transform.field.state-noise-std=0.05 "
               "--data.transform.model.view-aug --data.transform.model.view-aug-min-scale=0.85")
# CFG["EXTRA_ARGS"] = ROBUST_ARGS

In [ ]:
# @title 2. GPU, Google Drive, secrets
import datetime
import io
import os
import pathlib
import subprocess

from google.colab import auth, userdata
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

os.environ["NB_READY"] = ""  # set again at the end, once everything below is exported
os.environ.update(CFG)
os.environ["PSI_HOME"] = "/content/psi_home"
os.environ["RUNS_DIR"] = "/content/runs"  # training writes here; cell 9 uploads it to Drive

name, mem_mib, cc = [s.strip() for s in subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
    text=True).splitlines()[0].split(",")]
mem_gb, cc = int(mem_mib) / 1024, float(cc)
print(f"GPU: {name}, {mem_gb:.0f} GB, compute capability {cc}")
if cc < 8.0:
    raise RuntimeError("Need an Ampere or newer GPU (L4 / A100 / H100 / G4): training runs in bf16, "
                       "which a T4 does not have.")

# per-step batch by VRAM; gradient accumulation keeps the global batch at GLOBAL_BATCH
if CFG["BS"]:
    bs = int(CFG["BS"])
elif mem_gb >= 90:  # G4. The 12-block header doubles the trainable weights + AdamW state of box_pnp's
    bs = 32         # 6 blocks (~0.66 B params, ~10.6 GB in fp32 with the optimizer); cell 8 measures the peak
elif mem_gb >= 70:
    bs = 16
elif mem_gb >= 35:
    bs = 8
elif mem_gb >= 20:
    bs = 2
else:
    raise RuntimeError(f"{mem_gb:.0f} GB VRAM is too small for this recipe")
global_batch = int(CFG["GLOBAL_BATCH"])
assert global_batch % bs == 0, f"GLOBAL_BATCH {global_batch} is not a multiple of BS {bs}"
os.environ.update(BS=str(bs), ACC=str(global_batch // bs),
                  OPT_FOREACH="False" if mem_gb < 35 else "")  # non-foreach AdamW saves step memory
print(f"batch: BS={bs} x ACC={global_batch // bs} = {global_batch}")


class Drive:
    """My Drive of the account running this notebook, through the Drive API.

    Not drive.mount: a file deleted through the mount goes to the Drive trash and keeps counting
    against the quota (10 GB per rotated checkpoint), and the mount uploads in the background, so a
    checkpoint could look complete on Drive before all of it is there. Here upload() returns once
    the bytes are on Drive, and delete() is permanent.
    """
    FOLDER = "application/vnd.google-apps.folder"

    def __init__(self):
        auth.authenticate_user()
        self.svc = build("drive", "v3", cache_discovery=False)

    def children(self, folder_id):
        out, token = [], None
        while True:
            r = self.svc.files().list(q=f"'{folder_id}' in parents and trashed = false", pageSize=1000,
                                      pageToken=token, fields="nextPageToken, files(id, name, mimeType, size)"
                                      ).execute(num_retries=5)
            out += r["files"]
            token = r.get("nextPageToken")
            if not token:
                return out

    def child(self, folder_id, name):
        return next((f for f in self.children(folder_id) if f["name"] == name), None)

    def mkdir(self, parent, name):
        return self.svc.files().create(body={"name": name, "mimeType": self.FOLDER, "parents": [parent]},
                                       fields="id").execute(num_retries=5)["id"]

    def path(self, rel, create=False):
        """Id of the file or folder at `rel` (relative to My Drive), or None."""
        fid = "root"
        for part in pathlib.PurePosixPath(rel).parts:
            f = self.child(fid, part)
            if f is None and not create:
                return None
            fid = f["id"] if f else self.mkdir(fid, part)
        return fid

    def upload(self, local, parent, name=None):
        """Upload a file, or a directory recursively, into `parent`; replaces a file of the same name."""
        local = pathlib.Path(local)
        name = name or local.name
        if local.is_dir():
            fid = self.mkdir(parent, name)
            for p in sorted(local.iterdir()):
                if not p.is_symlink():
                    self.upload(p, fid)
            return fid
        size = local.stat().st_size
        old = self.child(parent, name)
        if size == 0:  # nothing to send; an empty upload body is rejected
            return old["id"] if old else self.svc.files().create(
                body={"name": name, "parents": [parent]}, fields="id").execute(num_retries=5)["id"]
        media = MediaFileUpload(str(local), resumable=size > 8 * 2**20, chunksize=256 * 2**20)
        if old:
            req = self.svc.files().update(fileId=old["id"], media_body=media, fields="id, size")
        else:
            req = self.svc.files().create(body={"name": name, "parents": [parent]}, media_body=media,
                                          fields="id, size")
        if media.resumable():
            resp = None
            while resp is None:
                _, resp = req.next_chunk(num_retries=5)
        else:
            resp = req.execute(num_retries=5)
        if int(resp.get("size", -1)) != size:
            raise IOError(f"{local}: Drive holds {resp.get('size')} bytes, expected {size}")
        return resp["id"]

    def download(self, file_id, local):
        local = pathlib.Path(local)
        local.parent.mkdir(parents=True, exist_ok=True)
        with open(local, "wb") as fh:
            dl = MediaIoBaseDownload(fh, self.svc.files().get_media(fileId=file_id), chunksize=64 * 2**20)
            done = False
            while not done:
                _, done = dl.next_chunk(num_retries=5)

    def download_tree(self, item, local):
        """`item`: an entry of children(); a folder is downloaded recursively."""
        if item["mimeType"] == self.FOLDER:
            for c in self.children(item["id"]):
                self.download_tree(c, pathlib.Path(local) / c["name"])
        else:
            self.download(item["id"], local)

    def read_text(self, rel):
        fid = self.path(rel)
        if fid is None:
            return ""
        buf = io.BytesIO()
        dl = MediaIoBaseDownload(buf, self.svc.files().get_media(fileId=fid))
        done = False
        while not done:
            _, done = dl.next_chunk(num_retries=5)
        return buf.getvalue().decode()

    def write_text(self, rel, text):
        rel = pathlib.PurePosixPath(rel)
        tmp = pathlib.Path("/tmp") / rel.name
        tmp.write_text(text)
        self.upload(tmp, self.path(str(rel.parent), create=True))

    def rename(self, file_id, name):
        self.svc.files().update(fileId=file_id, body={"name": name}).execute(num_retries=5)

    def delete(self, file_id):
        self.svc.files().delete(fileId=file_id).execute(num_retries=5)  # permanent: skips the trash

    def free_bytes(self):
        q = self.svc.about().get(fields="storageQuota").execute(num_retries=5)["storageQuota"]
        return int(q["limit"]) - int(q["usage"]) if "limit" in q else None


gd = Drive()
print("Google Drive: connected")
if gd.path(CFG["DATA_ZIP"]) is None:
    print(f"WARNING: My Drive/{CFG['DATA_ZIP']} not found - upload the zip written by export_psi0.py --zip")
free = gd.free_bytes()
if free is not None:
    print(f"Drive free: {free / 2**30:.0f} GB"
          + ("  (WARNING: < 40 GB - two checkpoints (~15-17 GB each) are on Drive while a new one uploads)"
             if free < 40 * 2**30 else ""))

try:
    wandb_key = userdata.get("WANDB_API_KEY")
except Exception:  # secret missing or notebook access not granted
    wandb_key = None
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
    os.environ.pop("WANDB_MODE", None)
    print("W&B: online")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("W&B: no WANDB_API_KEY secret -> offline (logs stay in the run dir)")

# the run id lives on Drive next to the runs, so a new VM resumes the same run
ts_rel = f"{CFG['DRIVE_RUNS']}/{CFG['EXP']}.run_ts"
run_ts = CFG["RUN_TS"] or gd.read_text(ts_rel).strip() or datetime.datetime.now().strftime("%y%m%d%H%M")
gd.write_text(ts_rel, run_ts)
os.environ["RUN_TS"] = run_ts
print(f"run: EXP={CFG['EXP']} RUN_TS={run_ts} (stored in My Drive/{ts_rel})")
os.environ["NB_READY"] = "1"

In [ ]:
%%bash
# 3. Environment: /content/Psi0 at PSI0_COMMIT + .venv-psi (Python 3.11, torch 2.7, flash-attn or SDPA)
set -euo pipefail
: "${NB_READY:?run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)}"
# Colab's own values would leak into the venv; UV_SYSTEM_PYTHON would point uv at Colab's python
unset PYTHONPATH MPLBACKEND UV_SYSTEM_PYTHON
export GIT_LFS_SKIP_SMUDGE=1 PATH="$HOME/.local/bin:$PATH"
# uv.lock at PSI0_COMMIT does not parse: its relock (6b4a5cc) left a duplicated table and packages
# listed under two registries. This is the last lock that parses, all on PyPI; the psi group is the
# same since, and the only newer dependencies (decord2, wandb[media]) are not used by this training.
GOOD_LOCK_COMMIT=47993ee31926235b2a640cc554a8c93f66220a3a
# the wheel scripts/train/Dockerfile uses: cu12 / torch 2.7 / cxx11abi TRUE / cp311, sm80 + sm90 kernels
FLASH_ATTN_WHEEL="https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.7cxx11abiTRUE-cp311-cp311-linux_x86_64.whl"
# Blackwell (G4 = RTX PRO 6000, compute capability 12.0): the locked torch 2.7.0 is the CUDA 12.6 build,
# which has no sm_100/sm_120 kernels, and flash-attn 2 publishes no Blackwell wheel. There the same
# torch 2.7.0 comes from the CUDA 12.8 index (README troubleshooting 5) and the VLM uses PyTorch's
# SDPA attention: the same attention, computed by a different kernel.
CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -n 1)
if [ "${CC%%.*}" -ge 10 ]; then VARIANT=cu128-sdpa; else VARIANT=cu126-flash; fi
echo "GPU compute capability $CC -> $VARIANT"

mkdir -p /content/Psi0 && cd /content/Psi0
[ -d .git ] || { git init -q && git remote add origin "$PSI0_REPO"; }
if [ "$(git rev-parse -q --verify HEAD || true)" != "$PSI0_COMMIT" ]; then
    git fetch -q --depth 1 origin "$PSI0_COMMIT"
    git checkout -q --force FETCH_HEAD
fi
echo "Psi0 @ $(git rev-parse --short HEAD)"

command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh
if [ -f .venv-psi/.synced ]; then
    echo "locked dependencies already installed"
else
    uv python install 3.11
    [ -d .venv-psi ] || uv venv .venv-psi --python 3.11
    source .venv-psi/bin/activate
    git fetch -q --depth 1 origin "$GOOD_LOCK_COMMIT"
    git show FETCH_HEAD:uv.lock > uv.lock
    # --frozen: install the lock as-is; --inexact: keep what is installed on top of it (flash-attn,
    # the CUDA 12.8 torch)
    uv sync --active --frozen --group psi --inexact
    touch .venv-psi/.synced
fi
# Checked against what is installed rather than a marker, so a half-finished run repairs itself.
if [ "$VARIANT" = cu128-sdpa ]; then
    if [ "$(.venv-psi/bin/python -c 'import torch; print(torch.version.cuda)')" != 12.8 ]; then
        # the +cu128 local version is required: a plain ==2.7.0 is already satisfied by the
        # locked 2.7.0+cu126, and uv would keep that build
        uv pip install --python .venv-psi/bin/python torch==2.7.0+cu128 torchvision==0.22.0+cu128 \
            --index-url https://download.pytorch.org/whl/cu128
    fi
elif ! .venv-psi/bin/python -c "import flash_attn" 2>/dev/null; then
    uv pip install --python .venv-psi/bin/python "$FLASH_ATTN_WHEEL"
fi
if [ "$VARIANT" = cu128-sdpa ]; then
    # finetune.py hard-codes flash_attention_2 for the VLM (the serving code already falls back to sdpa)
    sed -i 's/attn_implementation="flash_attention_2"/attn_implementation="sdpa"/' src/psi/trainers/finetune.py
fi
grep -n 'attn_implementation=' src/psi/trainers/finetune.py

source .venv-psi/bin/activate
python - "$VARIANT" <<'PY'
import sys, torch, psi
from psi.data.lerobot.compat import LEROBOT_LAYOUT
# fails here, not mid-training, when the build has no kernel for this GPU
x = torch.randn(256, 256, device="cuda", dtype=torch.bfloat16)
(x @ x).sum().item()
if sys.argv[1] == "cu126-flash":
    import flash_attn
    from flash_attn import flash_attn_func
    q = torch.randn(1, 64, 4, 64, device="cuda", dtype=torch.bfloat16)
    flash_attn_func(q, q, q)
    attn = f"flash_attn {flash_attn.__version__}"
else:
    q = torch.randn(1, 4, 64, 64, device="cuda", dtype=torch.bfloat16)
    torch.nn.functional.scaled_dot_product_attention(q, q, q).sum().item()
    attn = "sdpa"
print(f"torch {torch.__version__} (cuda {torch.version.cuda}) | {attn} | lerobot layout {LEROBOT_LAYOUT} | "
      f"{torch.cuda.get_device_name(0)}")
PY

In [ ]:
%%bash
# 4. .env: scripts/train.py asserts load_dotenv() finds one. Secrets stay in the process
# environment (cell 2) and are never written to disk.
set -euo pipefail
: "${NB_READY:?run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)}"
mkdir -p "$PSI_HOME"/data "$PSI_HOME"/cache/hf "$PSI_HOME"/cache/checkpoints
cat > /content/Psi0/.env <<EOF
PSI_HOME=$PSI_HOME
DATA_HOME=$PSI_HOME/data
HF_HOME=$PSI_HOME/cache/hf
TORCH_HOME=$PSI_HOME/cache/hf
HF_LEROBOT_HOME=$PSI_HOME/data/lerobot
OMP_NUM_THREADS=8
TOKENIZERS_PARALLELISM=false
CUDA_LAUNCH_BLOCKING=0
TF_CPP_MIN_LOG_LEVEL=3
AV_LOG_LEVEL=quiet
PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python
NO_ALBUMENTATIONS_UPDATE=1
EOF
cat /content/Psi0/.env

In [ ]:
# @title 5. Dữ liệu: tải từ Drive, giải nén, nạp thử một mẫu
import os
import pathlib
import shutil
import subprocess

assert os.environ.get("NB_READY"), "run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)"
data_root = pathlib.Path(os.environ["PSI_HOME"]) / "data/can_ped"
if not (data_root / ".unzipped").exists():
    zip_id = gd.path(CFG["DATA_ZIP"])
    if zip_id is None:
        raise FileNotFoundError(f"My Drive/{CFG['DATA_ZIP']} not found - upload the zip written by export_psi0.py --zip")
    print(f"downloading My Drive/{CFG['DATA_ZIP']} ...")
    gd.download(zip_id, "/content/pack.zip")
    shutil.rmtree(data_root, ignore_errors=True)
    data_root.mkdir(parents=True)
    subprocess.run(["unzip", "-q", "/content/pack.zip", "-d", str(data_root)], check=True)
    os.remove("/content/pack.zip")
    (data_root / ".unzipped").touch()

# checked with the training venv: same LeRobot class, video backend and delta-timestamp keys the loader uses
CHECK = """
import filecmp, json, sys
from pathlib import Path
root, pack = Path(sys.argv[1]), sys.argv[2]
train, val = root / f"{pack}_train", root / f"{pack}_val"
assert filecmp.cmp(train / "meta/stats_psi0.json", val / "meta/stats_psi0.json", shallow=False), \
    "train/val stats_psi0.json differ: both splits must share one normalization"
for d in (train, val):
    info = json.loads((d / "meta/info.json").read_text())
    names = info["features"]["observation.state"]["names"]
    # the SONIC 1.1 UniFoLM post-train packs use the 0810 joint order: leg(12) waist(3) arm(14) hand(14)
    assert (names[0], names[22], names[29]) == \
        ("left_hip_pitch_joint", "right_shoulder_pitch_joint", "left_hand_thumb_0_joint"), "state not in 0810 order"
    n = info["total_episodes"]
    assert len(list((d / "data/chunk-000").glob("*.parquet"))) == n, f"{d}: parquet count != {n}"
    assert len(list((d / "videos/chunk-000/observation.images.egocentric").glob("*.mp4"))) == n, f"{d}: video count != {n}"
    print(f"{d.name}: {n} episodes, {info['total_frames']} frames, "
          f"video {info['features']['observation.images.egocentric']['shape']}")
from psi.data.lerobot.compat import LeRobotDataset
chunk = [t / 30 for t in range(30)]
ds = LeRobotDataset(f"{pack}_train", root=train, video_backend="pyav",
                    delta_timestamps={"observation.images.egocentric": [0.0], "observation.state": [0.0],
                                      "action.body_token": chunk, "action": chunk})
sample = ds[len(ds) // 2]
print({k: tuple(v.shape) for k, v in sample.items() if hasattr(v, "shape") and v.ndim > 0})
print("instruction:", sample["task_description"])
"""
env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "MPLBACKEND")}
r = subprocess.run(["/content/Psi0/.venv-psi/bin/python", "-", str(data_root), CFG["PACK"]], input=CHECK,
                   text=True, capture_output=True, cwd="/content/Psi0", env=env)
print(r.stdout)
if r.returncode != 0:
    raise RuntimeError(f"data check failed:\n{r.stderr[-3000:]}")

In [ ]:
%%bash
# 6. Warm start: postpre.sonic1.1.unifolm.2609181726.40k, the SONIC 1.1 post-train on UniFoLM G1 data (no neck: action
# token 64 + hands 14 padded to 80, state 43 padded to 45, 12 action blocks). The Hugging Face folder is already in the
# exported layout (VLM files + action_header.safetensors), so both --model paths point at it as-is.
set -euo pipefail
: "${NB_READY:?run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)}"
unset PYTHONPATH MPLBACKEND
cd /content/Psi0 && source .venv-psi/bin/activate && set -a && source .env && set +a
CK=$PSI_HOME/cache/checkpoints
PT=psi0/postpre.sonic1.1.unifolm.2609181726.40k
INIT=$CK/$PT
if [ ! -s "$INIT/model.safetensors" ] || [ ! -s "$INIT/action_header.safetensors" ]; then
    # first run only, ~5-10 min; `du -sh $CK` in a Colab terminal shows the progress
    echo "[$(date +%T)] downloading $PT (11.2 GB: VLM 8.5 GB + action header 2.7 GB) ..."
    hf download USC-PSI-Lab/psi-model --local-dir "$CK" --include "$PT/*"
    echo "[$(date +%T)] download done"
fi
ls -la "$INIT"
python - "$INIT/action_header.safetensors" <<'PY'
# The header is loaded with strict=False, so a flag that changes its layout would silently re-initialise weights
# instead of failing. Check every shape the train command (cell 7) implies against the checkpoint file.
import re, sys
from safetensors import safe_open
with safe_open(sys.argv[1], "pt") as f:
    shape = {k: tuple(f.get_slice(k).get_shape()) for k in f.keys()}
blocks = {int(m.group(1)) for k in shape for m in [re.match(r"transformer_blocks\.(\d+)\.", k)] if m}
checks = {
    "action_dim 80": shape["action_proj_out.linear.weight"][0] == 80,
    "odim 45": shape["obs_proj._obs_proc.1.weight"][1] == 45,
    "action_chunk_size 30": shape["action_proj_in.dec_pos"][0] == 30,
    "num_blocks 12": len(blocks) == 12,
    "state_null_token": "state_null" in shape,
    "pooled CLIP text (768) -> combined_temb": shape.get("time_ins_embed.text_embedder.linear_1.weight", (0, 0))[1] == 768,
    "qk_norm": "transformer_blocks.0.attn.norm_q.weight" in shape,
    "view_feature_dim 2048": shape["obs_proj.views_proj.weight"][1] == 2048,
}
bad = [k for k, ok in checks.items() if not ok]
assert not bad, f"warm-start checkpoint does not match the train command: {bad}"
print(f"warm-start header matches the train command ({len(shape)} tensors, {len(blocks)} blocks)")
PY

In [ ]:
%%bash
# 7. Train launcher. Every knob comes from the environment (cells 1-2), so the smoke test and the real run share one
# command. Model / padding flags follow the SONIC 1.1 recipes of the Psi0 release (posttrain-psix-unifolm-g1-sonic1.1.sh,
# finetune-real-sonic-psi0-2.8B-sonic1.1-robust.sh), single GPU, VLM frozen (the release recipe tunes it on 8 GPUs).
cat > /content/finetune_canped.sh <<'EOF'
#!/bin/bash
set -euo pipefail
unset PYTHONPATH MPLBACKEND
cd /content/Psi0 && source .venv-psi/bin/activate
DATA_ROOT=$PSI_HOME/data/can_ped
INIT=$PSI_HOME/cache/checkpoints/psi0/postpre.sonic1.1.unifolm.2609181726.40k
echo "exp=$EXP ts=$RUN_TS out=$OUT_DIR bs=$BS acc=$ACC steps=$MAX_STEPS report_to=$REPORT_TO extra=${EXTRA_ARGS:-}"
# single process: accelerate runs without torchrun on one GPU
exec python scripts/train.py finetune_sonic_psi0_config \
    --seed=292285 --exp="$EXP" --timestamp="$RUN_TS" --train.resume_from_checkpoint=latest \
    --train.name=finetune --train.output_dir="$OUT_DIR" \
    --train.data_parallel=ddp --train.mixed_precision=bf16 \
    --train.train_batch_size="$BS" --train.gradient_accumulation_steps="$ACC" \
    --train.learning_rate=1e-4 --train.max_training_steps="$MAX_STEPS" \
    --train.warmup_ratio=None --train.warmup_steps="$WARMUP" \
    --train.checkpointing_steps="$CKPT_EVERY" --train.max_checkpoints_to_keep="$KEEP_CKPTS" \
    --train.preserve_checkpoint_every=0 \
    --train.validation_steps="$VAL_EVERY" --train.val_num_batches="$VAL_BATCHES" \
    --train.max_grad_norm=1.0 --train.lr_scheduler_type=cosine \
    --train.lr_scheduler_kwargs.weight_decay=1e-6 --train.lr_scheduler_kwargs.betas 0.95 0.999 \
    ${OPT_FOREACH:+--train.optimizer_foreach=$OPT_FOREACH} \
    --log.report_to="$REPORT_TO" \
    --data.root_dir="$DATA_ROOT" \
    --data.train_repo_ids="${PACK}_train" --data.val_repo_ids="${PACK}_val" \
    --data.transform.repack.image-keys observation.images.egocentric \
    --data.transform.repack.state-keys observation.state \
    --data.transform.repack.action-keys action.body_token 'action[:14]' \
    --data.transform.repack.action-chunk-size=30 \
    --data.transform.repack.dataset-name=can_ped \
    --data.transform.repack.pad-action-dim=80 --data.transform.repack.pad-state-dim=45 \
    --data.transform.repack.instruction-key=task_description \
    --data.transform.field.stat-path="$DATA_ROOT/${PACK}_train/meta/stats_psi0.json" \
    --data.transform.field.stat-state-keys observation.state \
    --data.transform.field.stat-action-keys action.body_token 'action[:14]' \
    --data.transform.field.action_norm_type=bounds --data.transform.field.normalize-state \
    --data.transform.field.pad-action-dim=80 --data.transform.field.pad-state-dim=45 \
    --data.transform.model.img-aug \
    --data.transform.model.resize.size 240 320 --data.transform.model.center_crop.size 240 320 \
    --model.model_name_or_path="$INIT" --model.pretrained-action-header-path="$INIT" \
    --model.noise-scheduler=flow --model.train-diffusion-steps=1000 --model.n_conditions=0 \
    --model.action-chunk-size=30 --model.action-dim=80 --model.action-exec-horizon=30 \
    --model.observation-horizon=1 --model.odim=45 --model.view_feature_dim=2048 \
    --model.dropout=0.0 --model.state-feature-dropout=0.0 \
    --model.no-tune-vlm --model.no-use_film \
    --model.qk-norm=rms_norm --model.combined-temb \
    --model.num-blocks=12 --model.vlm-layer-indices 3 5 8 10 12 14 17 19 21 23 26 28 \
    --model.state-drop-prob="$STATE_DROP_PROB" --model.state-as-action-token --model.state-null-token \
    --model.pooled-text-encoder=clip --model.pooled-text-encoder-path=openai/clip-vit-large-patch14 \
    --model.pooled-projection-dim=768 --model.pooled-cache-path=clip_pooled_cache.pt \
    --model.no-rtc --model.max-delay=8 \
    ${EXTRA_ARGS:-}
EOF
chmod +x /content/finetune_canped.sh
echo "wrote /content/finetune_canped.sh"

### Smoke test

Chạy 40 bước (ACC=1), 2 lần validation và 1 lần lưu checkpoint trên đĩa local. Mục đích: chứng minh cả chuỗi
(nạp dữ liệu, nạp warm-start, forward/backward, lưu checkpoint) chạy được, đo VRAM đỉnh và tốc độ.

- Nếu **OOM**: giảm `BS` ở cell 1 (ví dụ `"BS": "4"`), chạy lại cell 1, 2 rồi cell này.
- Trong log, dòng `[action header] ... not loaded as-is` hoặc `only loaded transformer blocks` nghĩa là warm-start không nạp đủ.
  Khi đó **không** chạy run thật.

In [ ]:
%%bash
# 8. Smoke test: 40 optimizer steps, val at 20 and 40, one checkpoint save, all on local disk
set -uo pipefail
: "${NB_READY:?run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)}"
rm -rf /content/smoke_runs
nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -l 1 > /content/vram.log &
SAMPLER=$!
EXP="smoke-$EXP" RUN_TS=smoke OUT_DIR=/content/smoke_runs ACC=1 MAX_STEPS=40 WARMUP=5 \
    CKPT_EVERY=40 VAL_EVERY=20 VAL_BATCHES=2 KEEP_CKPTS=1 REPORT_TO=None \
    bash /content/finetune_canped.sh > /content/smoke.log 2>&1
STATUS=$?
kill $SAMPLER
tr '\r' '\n' < /content/smoke.log | grep -v '^\s*$' | tail -n 25
echo "----"
grep -E "not loaded as-is|only loaded transformer blocks|size mismatch" /content/smoke.log \
    && echo "WARNING: warm-start was not fully loaded (see above)"
echo "exit status: $STATUS | peak VRAM: $(sort -n /content/vram.log | tail -1) MiB"
rm -rf /content/smoke_runs
exit $STATUS

### Train

Chạy cell 9 và **để nó chạy suốt**. Nó khởi động train ở tiến trình nền (hoặc chỉ theo dõi nếu train đang chạy), rồi mỗi phút cập nhật:
bước hiện tại, loss, GPU, và checkpoint mới nhất đã lên Drive. Mỗi checkpoint được tải lên `My Drive/<DRIVE_RUNS>/finetune/<run>/checkpoints/`
dưới tên `ckpt_N.partial`, chỉ đổi thành `ckpt_N` khi đã lên đủ, rồi checkpoint cũ hơn mới bị xoá (xoá hẳn, không qua Thùng rác).
Cell tự kết thúc khi train xong và checkpoint cuối đã lên Drive.

In [ ]:
# @title 9. Train: để cell này chạy suốt quá trình train
# Starts training in the background (unless it is already running), then loops until it ends:
# shows progress every minute and uploads each finished checkpoint to Drive. Interrupting this
# cell does not stop training; run it again to resume watching and uploading.
import os
import pathlib
import re
import subprocess
import time

from IPython.display import clear_output

assert os.environ.get("NB_READY"), "run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)"
LOCAL = pathlib.Path(os.environ["RUNS_DIR"])
RUN_TS = os.environ["RUN_TS"]
LOG = LOCAL / "logs" / f"{CFG['EXP']}.{RUN_TS}.log"
LOGS_REL = f"{CFG['DRIVE_RUNS']}/logs"


def training_running():
    return subprocess.run(["pgrep", "-f", "scripts/train.py"], stdout=subprocess.DEVNULL).returncode == 0


def ckpt_step(name):
    m = re.fullmatch(r"ckpt_(\d+)", name)
    return int(m.group(1)) if m else None


def local_run_dir():
    return next(iter(sorted(LOCAL.glob(f"finetune/*.{RUN_TS}"))), None)


def drive_run_dir(create_name=None):
    """This run's folder on Drive (matched by RUN_TS), created as `create_name` if given and missing."""
    finetune = gd.path(f"{CFG['DRIVE_RUNS']}/finetune", create=create_name is not None)
    if finetune is None:
        return None
    for f in gd.children(finetune):
        if f["mimeType"] == gd.FOLDER and f["name"].endswith(f".{RUN_TS}"):
            return f
    if create_name is None:
        return None
    return {"id": gd.mkdir(finetune, create_name), "name": create_name, "mimeType": gd.FOLDER}


def drive_ckpts(run):
    """(checkpoints folder, {step: entry} of its complete checkpoints) of a Drive run folder."""
    ck = gd.child(run["id"], "checkpoints") if run else None
    if ck is None:
        return None, {}
    return ck, {ckpt_step(f["name"]): f for f in gd.children(ck["id"]) if ckpt_step(f["name"]) is not None}


def finished_ckpts(alive):
    """Local checkpoints that are completely written, oldest first."""
    run = local_run_dir()
    out = []
    for d in (run / "checkpoints").glob("ckpt_*") if run else []:
        step = ckpt_step(d.name)
        if step is None or not (d / "random_states_0.pkl").exists():  # accelerate's save_state writes it last
            continue
        if alive and time.time() - max(p.stat().st_mtime for p in d.rglob("*")) < 60:
            continue  # still being written
        out.append((step, d))
    return sorted(out)


def restore():
    """New VM: bring back this run's newest complete checkpoint, its run files and its log from Drive."""
    if local_run_dir() is not None:
        return
    run = drive_run_dir()
    _, done = drive_ckpts(run)
    if not done:
        print("no checkpoint of this run on Drive: training starts from step 0")
        return
    step = max(done)
    dst = LOCAL / "finetune" / run["name"]
    print(f"restoring ckpt_{step} from Drive (~15-17 GB) ...")
    for f in gd.children(run["id"]):
        if f["mimeType"] != gd.FOLDER:
            gd.download(f["id"], dst / f["name"])
    gd.download_tree(done[step], dst / "checkpoints" / f"ckpt_{step}")
    log_id = gd.path(f"{LOGS_REL}/{LOG.name}")
    if log_id:
        gd.download(log_id, LOG)


def upload_ckpt(step, d):
    """Upload as ckpt_N.partial, rename once every byte is on Drive, then drop the older ones."""
    run = drive_run_dir(create_name=d.parent.parent.name)
    ck, _ = drive_ckpts(run)
    if ck is None:
        ck = {"id": gd.mkdir(run["id"], "checkpoints")}
    for f in gd.children(ck["id"]):
        if f["name"].endswith(".partial"):  # left by an interrupted upload
            gd.delete(f["id"])
    gd.rename(gd.upload(d, ck["id"], f"ckpt_{step}.partial"), f"ckpt_{step}")
    for f in gd.children(ck["id"]):
        old = ckpt_step(f["name"])
        if old is not None and old < step:
            gd.delete(f["id"])
    return step


def upload_small():
    """The run's top-level files (run_config.json, argv.txt, dataset_statistics.json, ...) and the log."""
    run = local_run_dir()
    if run is not None:
        drive_run = drive_run_dir(create_name=run.name)
        for p in sorted(run.iterdir()):
            if p.is_file() and not p.is_symlink():
                gd.upload(p, drive_run["id"])
    if LOG.exists():
        gd.upload(LOG, gd.path(LOGS_REL, create=True))


def show(alive, on_drive, note=""):
    text = LOG.read_text(errors="ignore")[-6000:].replace("\r", "\n") if LOG.exists() else ""
    lines = [l.rstrip() for l in text.splitlines() if l.strip()]
    progress = next((l for l in reversed(lines) if "Training steps" in l), "(no progress line yet)")
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    clear_output(wait=True)
    print(f"{time.strftime('%F %H:%M:%S')} (Colab clock, UTC) | training {'RUNNING' if alive else 'NOT RUNNING'} | GPU {gpu}")
    print(progress)
    print(f"checkpoint on Drive: {f'ckpt_{on_drive}' if on_drive is not None else 'none yet'} "
          f"(My Drive/{CFG['DRIVE_RUNS']}/finetune)")
    if note:
        print(note)
    if not alive:
        print("---- end of the log\n" + "\n".join(lines[-15:]))


# ---- main
max_steps = int(os.environ["MAX_STEPS"])
if not training_running():
    restore()
    if max((s for s, _ in finished_ckpts(False)), default=-1) >= max_steps:
        print(f"this run already reached step {max_steps}: nothing to train")
    else:
        LOG.parent.mkdir(parents=True, exist_ok=True)
        with open(LOG, "a") as log:
            log.write(f"=== launch {time.strftime('%F %T')} ===\n")
            log.flush()
            subprocess.Popen(["bash", "/content/finetune_canped.sh"], stdin=subprocess.DEVNULL, stdout=log,
                             stderr=subprocess.STDOUT, start_new_session=True,
                             env={**os.environ, "OUT_DIR": str(LOCAL), "REPORT_TO": "wandb"})
        print("training started, log:", LOG)
        time.sleep(20)

on_drive, last_small, note, fin = None, 0.0, "", []
while True:
    alive = training_running()
    try:
        if on_drive is None:
            on_drive = max(drive_ckpts(drive_run_dir())[1], default=None)
        fin = finished_ckpts(alive)
        if fin and (on_drive is None or fin[-1][0] > on_drive):
            show(alive, on_drive, f"uploading ckpt_{fin[-1][0]} to Drive (~15-17 GB, a few minutes) ...")
            on_drive = upload_ckpt(*fin[-1])
            last_small = 0.0
        if not alive or time.time() - last_small > 600:
            upload_small()
            last_small = time.time()
        note = ""
    except Exception as e:  # network / Drive API trouble: training goes on, retry next minute
        note = f"Drive upload failed, retrying in a minute: {e!r}"
    show(alive, on_drive, note)
    if not alive and not note:
        break
    if not alive and fin and fin[-1][0] == on_drive:
        break
    time.sleep(60)
print("training is not running; newest checkpoint on Drive:", f"ckpt_{on_drive}" if on_drive is not None else "none")

In [ ]:
# @title 10. Dừng train
# Stop cell 9 first (it keeps running while training does), set STOP_TRAINING = True, run this cell.
# The newest finished checkpoint is uploaded before the cell returns; run cell 9 to resume later.
import subprocess
import time

STOP_TRAINING = False
if not STOP_TRAINING:
    print("STOP_TRAINING is False: nothing stopped")
elif subprocess.run(["pkill", "-TERM", "-f", "scripts/train.py"]).returncode != 0:
    print("no training process")
else:
    for _ in range(60):
        time.sleep(2)
        if subprocess.run(["pgrep", "-f", "scripts/train.py"], stdout=subprocess.DEVNULL).returncode != 0:
            break
    print("training stopped")
    if "upload_ckpt" in globals():
        fin = finished_ckpts(False)
        have = max(drive_ckpts(drive_run_dir())[1], default=None)
        if fin and (have is None or fin[-1][0] > have):
            print(f"uploading ckpt_{fin[-1][0]} to Drive ...")
            upload_ckpt(*fin[-1])
        upload_small()
        print("Drive is up to date")

### Sau khi train xong

- Checkpoint cuối: `My Drive/<DRIVE_RUNS>/finetune/<EXP>...gpus1.<RUN_TS>/checkpoints/ckpt_<MAX_STEPS>/`. Thư mục run cũng chứa
  `run_config.json` và `dataset_statistics.json`; tải cả thư mục run về là server (`serve_psi0_sonic_http --run-dir <run> --ckpt-step <N>`)
  đọc được. Trong checkpoint, `model.safetensors` là trọng số model; `optimizer.bin` chỉ cần để train tiếp.
- **Server phải chạy code Ψ₀ `2830f93`** (bản có `combined-temb`, CLIP pooled text, `state-as-action-token`); code `4f3720d` của notebook
  server `box_pnp` không dựng được model này.
- Model xuất action **80 chiều**: `token 64 | tay 14 | 2 chiều đệm` (cổ, không dùng). `can_pnp/closed_loop_eval.py` lấy 78 chiều đầu.
  Server tự đệm state 43 → 45, client vẫn gửi 43.
- Model xuất token **SONIC 1.1**: khi chạy thử hay deploy, decoder phải là `policy/sonic_v1_1` (mặc định của `closed_loop_eval.py`).
  Trước khi gửi cho SONIC, client snap token về lưới FSQ (bước 1/16).
- W&B offline: log nằm trong thư mục run trên Colab. Đồng bộ sau bằng `wandb sync` khi đã có API key.